In [9]:
%run ./imports.ipynb
%run ./tools_fun.ipynb
%run ./tools_handle.ipynb
%run ./db_config.ipynb

In [10]:
load_dotenv(override=True)
google_api_key = os.getenv('GOOGLE_API_KEY')
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
model = "gemini-2.5-flash"

gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)


In [11]:
load_dotenv(override=True)
google_api_key = os.getenv('GOOGLE_API_KEY')

# Add this debug line to check if it's working
if not google_api_key:
    print("ERROR: GOOGLE_API_KEY not found in environment variables!")
else:
    print("API Key loaded successfully.")

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
model = "gemini-2.5-flash"

# The error happens here because google_api_key is None
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)

API Key loaded successfully.


In [12]:
system_message = """
You are FlightAI's premium booking assistant. 
1. Check prices using get_ticket_price.
2. If the user wants to book, ask for their full name if they haven't provided it.
3. Once you have the name and destination, use book_flight.
Be professional, concise, and confirm all details before booking.
"""

In [13]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    
    response = gemini.chat.completions.create(
        model=model,
        messages=messages,
        tools=tools
    )

    if response.choices[0].finish_reason == "tool_calls":
        assistant_message = response.choices[0].message
        
        # 2. Get the list of tool responses
        tool_responses = handle_tool_calls(assistant_message)
        
        messages.append(assistant_message)
        messages.extend(tool_responses) 
        
        response = gemini.chat.completions.create(
            model=model, 
            messages=messages
        )
    
    final_content = response.choices[0].message.content
    
    return final_content if final_content is not None else "Operation completed."

In [14]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


DATABASE TOOL CALLED: Searching bookings for Alex
